# Making an LLM API Query: One Question, One Response

This short notebook shows the basic parts of an application that sends text to an AI model:

1. Connect to the OpenAI API.
2. Give the model a **system prompt** describing how it should behave.
3. Collect a **user prompt**.
4. Send both prompts to GPT-5.6.
5. Print the model's response.

This is a **single-turn** application. It does not yet remember earlier questions. We will add conversation state in the later AI-investor activity.

## Before beginning: add the API key to Colab Secrets

1. Click the **🔑 Secrets** tab on the left side of Google Colab.
2. Add a secret named exactly `OPENAI_API_KEY`.
3. Paste the temporary camp API key as the value.
4. Enable **Notebook access** for the secret.
5. Never type the key into a code cell or print it.

The temporary key will be revoked after the camp activity.

In [ ]:
#@title 1. Install the modern OpenAI Python library
!pip -q install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.5 MB/s eta 0:00:00


In [ ]:
#@title 2. Connect to OpenAI using Colab Secrets

from google.colab import userdata
from openai import OpenAI

try:
    api_key = userdata.get("OPENAI_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Colab could not read OPENAI_API_KEY. "
        "Add it in the Secrets tab and enable Notebook access."
    ) from error

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is empty. "
        "Add the temporary key in the Colab Secrets tab."
    )

client = OpenAI(api_key=api_key)

print("OpenAI connection is ready.")

OpenAI connection is ready.
Model: gpt-5.6


## The system prompt

The system prompt gives the model its role and its general rules. The user does not need to repeat these rules in every question.

In [ ]:
SYSTEM_PROMPT = """
You are a helpful AI assistant in a supervised high-school educational camp.

Answer the student's question clearly and accurately using language appropriate
for high-school students. Keep the answer concise.

Do not request unnecessary personal information. If a request is clearly
inappropriate, briefly refuse and suggest a safe, educational alternative.
"""

print(SYSTEM_PROMPT)


You are a helpful AI assistant in a supervised high-school educational camp.

Answer the student's question clearly and accurately using language appropriate
for high-school students. Keep the answer concise.

Do not request unnecessary personal information. If a request is clearly
inappropriate, briefly refuse and suggest a safe, educational alternative.



## The function that contacts the model

The function below builds a request containing:

- the selected model,
- the system instructions,
- the user's input,
- and a limit on the response length.

It prints the request so that we can inspect what the program sends. The API key is never included in this printed dictionary.

In [ ]:
def generate_response(user_prompt):
    """Send one user prompt to OpenAI and return the model's text response."""

    request = {
        "model": "gpt-5.6",
        "instructions": SYSTEM_PROMPT,
        "input": user_prompt,
        "max_output_tokens": 300,
        "store": False
    }

    response = client.responses.create(**request)
    return response.output_text

## Ask one question

Enter an appropriate question below. The program will print the request and then the model's response.

In [ ]:
user_prompt = input("Enter your question: ").strip()

print("\nGenerating response...")

try:
    answer = generate_response(user_prompt)

    print("\nAI RESPONSE: ")
    print(answer)

except Exception as error:
    print("The request could not be completed.")
    print("Ask the instructor for help.")
    print("Technical error:", error)

Enter your question: How to split training and test datasets in python

Generating response...

AI RESPONSE: 
You can split data into training and test sets using **scikit-learn**:

```python
from sklearn.model_selection import train_test_split

# X contains input features; y contains target labels
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,    # 20% test data, 80% training data
    random_state=42   # Makes the split reproducible
)
```

For classification, keep class proportions similar in both sets:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

Install scikit-learn if needed:

```bash
pip install scikit-learn
```

Avoid using the test set during training; use it only for the final evaluation.


## Optional: ask a second independent question

This cell makes another one-turn request. The model receives the system prompt and the new question, but it does **not** receive the earlier question or answer.

In [ ]:
second_prompt = input("Enter a different question: ").strip()

print("\nGenerating a new independent response...")

try:
    second_answer = generate_response(second_prompt)

    print("\nAI RESPONSE: ")
    print(second_answer)

except Exception as error:
    print("The request could not be completed.")
    print("Ask the instructor for help.")
    print("Technical error:", error)

Enter a different question: Suggest an AI project idea

Generating a new independent response...

AI RESPONSE: 
### AI Project Idea: Smart Study Planner

Build an app that creates a personalized study schedule based on:

- Subjects and upcoming deadlines  
- How difficult each topic feels  
- Available study time  
- Quiz results or completed tasks  

**AI component:** Train a simple model to predict which subjects need more practice, then give them higher priority in the schedule.

**Tools:** Python, pandas, scikit-learn, and Streamlit for the interface.

**Extra features:** Progress charts, reminders, and automatic schedule adjustments when a task is missed.
